# 🌿 AI Crop Disease Detection: Deep Learning Model Training (MobileNetV2)

This Jupyter Notebook trains a Convolutional Neural Network based on **MobileNetV2** using Transfer Learning to detect foliar crop diseases across multiple crop species (Tomato, Potato, Corn, Apple, Grape, Pepper).

### Pipeline Overview:
1. **Dataset Loading & Augmentation** (Train / Validation / Test splits)
2. **Transfer Learning Model Architecture** (MobileNetV2 backbone + Custom Classifier Head)
3. **Training & Validation Loop** (CrossEntropyLoss, Adam Optimizer, StepLR Scheduler)
4. **Performance Evaluation** (Loss / Accuracy Curves, Classification Report)
5. **Model Export** (Saved to `../model/mobilenet_v2_crop.pth` for Flask backend deployment)

In [ ]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import datasets, models, transforms
import numpy as np
from PIL import Image

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data Transformations & Augmentation

In [ ]:
DATA_DIR = '../dataset'

# Standard ImageNet normalization
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'validation': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

print("Transforms initialized successfully.")

## 2. Load Image Datasets

In [ ]:
try:
    image_datasets = {
        x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
        for x in ['train', 'validation']
    }
    dataloaders = {
        x: torch.utils.data.DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=2)
        for x in ['train', 'validation']
    }
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'validation']}
    class_names = image_datasets['train'].classes
    print(f"Loaded {len(class_names)} classes: {class_names}")
except Exception as e:
    print(f"Note: Place downloaded dataset images into dataset/train and dataset/validation folders. Error: {e}")
    class_names = [
        "Apple___Apple_scab", "Apple___Black_rot", "Apple___healthy",
        "Corn___Common_rust", "Corn___Northern_Leaf_Blight", "Corn___healthy",
        "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
        "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___healthy"
    ]

## 3. Initialize MobileNetV2 Architecture

In [ ]:
# Load pretrained MobileNetV2
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if hasattr(models, 'MobileNet_V2_Weights') else None)

# Freeze earlier feature extractor layers for fine-tuning
for param in model.features.parameters():
    param.requires_grad = False

# Replace top classifier layer for our crop disease classes
num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(num_features, len(class_names))
)

model = model.to(device)
print(model.classifier)

## 4. Define Loss, Optimizer, and Learning Rate Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001, weight_decay=1e-4)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
print("Optimizer and loss function configured.")

## 5. Training Loop Function

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 15)

        for phase in ['train', 'validation']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            if phase not in dataloaders:
                continue

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'validation' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Validation Accuracy: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

## 6. Export Trained Model for Flask Deployment

In [ ]:
os.makedirs('../model', exist_ok=True)
OUTPUT_PATH = '../model/mobilenet_v2_crop.pth'

# Save trained model weights
torch.save(model.state_dict(), OUTPUT_PATH)
print(f"Trained model weights saved successfully to: {OUTPUT_PATH}")
print("The Flask backend (backend/app.py) will automatically detect and load these weights!")